In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
nx = 130
ny = 130
Lx = 1
Ly = 1

Ustar = 1
Re = 100

visc = Ustar*Lx/Re
CFL = 0.2 #dt = 0.005

endTime = 10

In [3]:
xL = 0
xR = xL+Lx
yB = 0
yT = yB+Ly

In [4]:
np = nx*ny
nu = ny*(nx-1) + nx*(ny-1)
dx = Lx/nx
dy = Ly/ny
dt = CFL*dx

In [6]:
def Func_GenPointer(nx,ny):
    ip = np.empty(nx,ny)
    iu = np.empty(nx,ny)
    iv = np.empty(nx,ny)
    
    count = 1
    for i in range(1,nx):
        for j in range(1,ny):
            ip[i,j] = count
            count = count+1
    
    count = 1
    for i in range(2,nx):
        for j in range(1,ny):
            iu[i,j] = count
            count = count+1
    
    count = 1    
    for i in range(1,nx):
        for j in range(2,ny):
            iv[i,j] = count
            count = count+1

    return [ip,iu,iv] 

def Func_GenGrid(xL,xR,nx,dx,yB,yT,ny,dy):
    xp = np.linspace(xL+dx/2,xR-dx/2,nx)
    yp = np.linspace(yB+dy/2,yT-dx/2,ny)
    
    xu = np.linspace(xL+dx/2,xR-dx/2,nx)
    yu = np.linspace(yB+dy/2,yT-dx/2,ny)
    
    xv = np.linspace(xL+dx/2,xR-dx/2,nx)
    yv = np.linspace(yB+dy/2,yT-dx/2,ny)
    
    return [xp,yp,xu,yu,xv,yv] 

In [ ]:
def Opt_D(x,ip,iu,iv):
    # x = input as velocity
    sol = np.zeros(np,1)
    #% inner domain
    for j in range(2,ny-1):
        for i in range(2,nx-1):
            sol[ip[i,j]] = (x[iu[i+1,j]] - x[iu[i,j]])/dx + (x[iv[i,j+1]] - x[iv[i,j]])/dy
            
        
    return sol 

def Opt_G(x):

    #x = input as pressure
    Grad_P = np.zeros(nu,1)

    return sol 

In [ ]:
def Laplace_x(u, Nx, Ny, index, dx, dy):
    value = np.zeros_like(u)
    # internal nodes
    for j in range(1, Ny-1):
        for i in range(1, Nx-1):
            idx = index[i, j]
            value[idx] = (u[index[i+1, j]] - 2*u[idx] + u[index[i-1, j]]) / dx**2 + \
                         (u[index[i, j+1]] - 2*u[idx] + u[index[i, j-1]]) / dy**2
    # west boundary (dirichlet)
    i = 0
    for j in range(1, Ny-1):
        idx = index[i, j]
        value[idx] = (-2*u[idx] + u[index[i+1, j]]) / dx**2 + \
                      + (u[index[i, j+1]] - 2*u[idx] + u[index[i, j-1]]) / dy**2
    # east boundary (neumann)
    i = Nx-1
    for j in range(1, Ny-1):
        idx = index[i, j]
        value[idx] = (u[index[i-1, j]] - u[idx]) / dx**2 + \
                      + (u[index[i, j+1]] - 2*u[idx] + u[index[i, j-1]]) / dy**2
    # north boundary (dirichlet)
    j = Ny-1  
    for i in range(1, Nx-1):
        idx = index[i, j]
        value[idx] = (u[index[i+1, j]] - 2*u[idx] + u[index[i-1, j]]) / dx**2 + \
                      + (-2*u[idx] + u[index[i, j-1]]) / dy**2
    # south boundary (neumann)
    j = 0
    for i in range(1, Nx-1):
        idx = index[i, j]
        value[idx] = (u[index[i+1, j]] - 2*u[idx] + u[index[i-1, j]]) / dx**2 + \
                      + (u[index[i, j+1]] - u[idx]) / dy**2
    # corners
    # southwest corner (west dirichlet + south neumann)
    i, j = 0, 0
    idx = index[i, j]
    value[idx] = (-2*u[idx] + u[index[i+1, j]]) / dx**2 + \
                  + (u[index[i, j+1]] - u[idx]) / dy**2
    # southeast corner (east neumann + south neumann)
    i, j = Nx-1, 0
    idx = index[i, j]
    value[idx] = (u[index[i-1, j]] - u[idx]) / dx**2 + \
                  + (u[index[i, j+1]] - u[idx]) / dy**2
    # northwest corner (west dirichlet + north dirichlet)
    i, j = 0, Ny-1
    idx = index[i, j]
    value[idx] = (-2*u[idx] + u[index[i+1, j]]) / dx**2 + \
                  + (-2*u[idx] + u[index[i, j-1]]) / dy**2
    # northeast corner (east neumann + north dirichlet)
    i, j = Nx-1, Ny-1
    idx = index[i, j]
    value[idx] = (u[index[i-1, j]] - u[idx]) / dx**2 + \
                  + (-2*u[idx] + u[index[i, j-1]]) / dy**2
    return value

In [ ]:
def Laplace_BC(Nx, Ny, index, dx, dy, BCL_D, BCR_N, BCB_N, BCT_D):
    value = np.zeros(Nx*Ny)

    # west boundary (dirichlet)
    i = 0
    for j in range(1, Ny-1):
        idx = index[i, j]
        value[idx] = BCL_D[j] / dx**2

    # east boundary (neumann)
    i = Nx-1
    for j in range(1, Ny-1):
        idx = index[i, j]
        value[idx] = BCR_N[j] / dx  

    # north boundary (dirichlet)
    j = Ny-1
    for i in range(1, Nx-1):
        idx = index[i, j]
        value[idx] = BCT_D[i] / dy**2

    # south boundary (neumann)
    j = 0
    for i in range(1, Nx-1):
        idx = index[i, j]
        value[idx] = -BCB_N[i] / dy  

    # corners
    # southwest corner (west dirichlet + south neumann)
    i, j = 0, 0
    idx = index[i, j]
    value[idx] = BCL_D[j] / dx**2 - BCB_N[i] / dy

    # southeast corner (east neumann + south neumann)
    i, j = Nx-1, 0
    idx = index[i, j]
    value[idx] = BCR_N[j] / dx - BCB_N[i] / dy 

    # northwest corner (west dirichlet + north dirichlet)
    i, j = 0, Ny-1
    idx = index[i, j]
    value[idx] = BCL_D[j] / dx**2 + BCT_D[i] / dy**2

    # northeast corner (east neumann + north dirichlet)
    i, j = Nx-1, Ny-1
    idx = index[i, j]
    value[idx] = BCR_N[j] / dx + BCT_D[i] / dy**2 

    return value

In [ ]:
[ip,iu,iv] = Func_GenPointer(nx,ny)
[xp,vp,xu,vu,xv,yv] = Func_GenGrid(xL,xR,nx,dx,yB,yT,ny,dy)

In [ ]:
uBC_T = Ustar*np.ones(1,nx)
uBC_B = np.zeros(1,nx)
uBC_L = np.zeros(ny,1)# column vector
uBC_R = np.zeros(ny,1)

vBC_T = Ustar*np.ones(1,nx)
vBC_B = np.zeros(1,nx)
vBC_L = np.zeros(ny,1)
vBC_R = np.zeros(ny,1)

In [ ]:
Opt_R = lambda x: x - (dt/2*visc)*Laplace_x(x)
Opt_S = lambda x: x + (dt/2*visc)*Laplace_x(x)
Opt_T = lambda x: Opt_D(Opt_S(Opt_G(x)))
Opt_C = lambda x: Opt_S(Opt_G(x))
BC_L = Laplace_BC
BC_D = BC_Div


In [12]:
test = [10 + 10 +
     14]

In [ ]:
# ==== Advection operator ====
def Adv(Qi):
    # global nu iu iv nx ny dx dy
    # global uBC_T uBC_B uBC_L uBC_R vBC_T vBC_B vBC_L vBC_R
    # advection operator (BC embedded): -\nabla \cdot (uu)
    # input: u-type (nu elements)
    # output: u-type (nu elements)
    #
    # Input size check:
    # if Qi.shape[1] ~= 1
    # error('Adv:: Need a colume vector input!!')
    # 


    # if size(Qi, 1) ~= nu
    # error('Adv:: input vector size mismatch!!')
    # 


    # Initialize output
    Qo = np.empty(nu, 1) 
    # 1. U-Component
    # inner domain
    for i in range(3,1,(nx-1)):
        for j in range(2,1,(ny-1)):
            Qo[iu[i, j]] = [- ( - ( Qi[iu[i-1,j]] + Qi[iu[i ,j]] ) * ( Qi[iu[i-1,j]] + Qi[iu[i ,j]] ) / 4 + 
                               (Qi[iu[i,j]] + Qi[iu[i+1,j]] ) * ( Qi[iu[i ,j]] + Qi[iu[i+1,j]] ) / 4 ) / dx - 
                               ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                                ( Qi[iu[i,j]] + Qi[iu[i,j+1]] ) * ( Qi[iv[i-1,j+1]] + Qi[iv[i ,j+1]] ) / 4 ) / dy] 
        


    


    # Edges
    # bottom inner
    j = 1 
    for i in range(3,1,(nx-1)):
        Qo[iu[i, j]] = [- ( - ( Qi[iu[i-1,j ]] + Qi[iu[i ,j]] ) * ( Qi[iu[i-1,j]] + Qi[iu[i ,j]] ) / 4  +
                           ( Qi[iu[i ,j]] + Qi[iu[i+1,j ]] ) * ( Qi[iu[i ,j]] + Qi[iu[i+1,j ]] ) / 4 ) / dx - 
                           ( - ( uBC_B(i) ) * ( vBC_B(i-1) + vBC_B(i) ) / 2 + 
                            ( Qi[iu[i ,j]] + Qi[iu[i ,j+1]] ) * ( Qi[iv[i-1,j+1]] + Qi[iv[i ,j+1]] ) / 4 ) / dy ]
    


    # top inner
    j = ny 
    for i in range(3,1,(nx-1)):
        Qo[iu[i, j]] = [- ( - ( Qi[iu[i-1,j ]] + Qi[iu[i ,j]] ) * ( Qi[iu[i-1,j]] + Qi[iu[i ,j]] ) / 4 + 
                           ( Qi[iu[i ,j]] + Qi[iu[i+1,j ]] ) * ( Qi[iu[i ,j]] + Qi[iu[i+1,j ]] ) / 4 ) / dx - 
                           ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                            ( uBC_T(i) ) * ( vBC_T(i-1) + vBC_T(i) ) / 2 ) / dy ]
    


    # left inner
    i = 2 
    for j in range(2,1,(ny-1)):
        Qo[iu[i, j]] = [- ( - ( uBC_L(j) + Qi[iu[i ,j]] ) * ( uBC_L(j) + Qi[iu[i ,j]] ) / 4 + 
                           ( Qi[iu[i ,j]] + Qi[iu[i+1,j ]] ) * ( Qi[iu[i ,j]] + Qi[iu[i+1,j ]] ) / 4 ) / dx - 
                           ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                            ( Qi[iu[i ,j]] + Qi[iu[i ,j+1]] ) * ( Qi[iv[i-1,j+1]] + Qi[iv[i ,j+1]] ) / 4 ) / dy ]
    


    # right inner
    i = nx 
    for j in range(2,1,(ny-1)):
        Qo[iu[i, j]] = [- ( - ( Qi[iu[i-1,j ]] + Qi[iu[i ,j]] ) * ( Qi[iu[i-1,j]] + Qi[iu[i ,j]] ) / 4  + 
                           ( Qi[iu[i ,j]] + uBC_R(j) ) * ( Qi[iu[i ,j]] + uBC_R(j) ) / 4 ) / dx - 
                           ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                            ( Qi[iu[i ,j]] + Qi[iu[i ,j+1]] ) * ( Qi[iv[i-1,j+1]] + Qi[iv[i ,j+1]] ) / 4 ) / dy ]
    


    # Corners
    # bottom left
    i = 2 
    j = 1 
    Qo[iu[i, j]] = [- ( - ( uBC_L(j) + Qi[iu[i ,j]] ) * ( uBC_L(j) + Qi[iu[i ,j]] ) / 4 + 
                       ( Qi[iu[i ,j]] + Qi[iu[i+1,j ]] ) * ( Qi[iu[i ,j]] + Qi[iu[i+1,j ]] ) / 4 ) / dx  - 
                       ( - ( uBC_B(i) ) * ( vBC_B(i-1) + vBC_B(i) ) / 2  + 
                        ( Qi[iu[i ,j]] + Qi[iu[i ,j+1]] ) * ( Qi[iv[i-1,j+1]] + Qi[iv[i ,j+1]] ) / 4 ) / dy ]

    # bottom right
    i = nx 
    j = 1 
    Qo[iu[i, j]] = [- ( - ( Qi[iu[i-1,j ]] + Qi[iu[i ,j]] ) * ( Qi[iu[i-1,j]] + Qi[iu[i ,j]] ) / 4  + 
                       ( Qi[iu[i ,j]] + uBC_R(j) ) * ( Qi[iu[i ,j]] + uBC_R(j) ) / 4 ) / dx  - 
                       ( - ( uBC_B(i) ) * ( vBC_B(i-1)+ vBC_B(i) ) / 2 + 
                        ( Qi[iu[i ,j]] + Qi[iu[i ,j+1]] ) * ( Qi[iv[i-1,j+1]] + Qi[iv[i ,j+1]] ) / 4 ) / dy ]

    # top left
    i = 2 
    j = ny 
    Qo[iu[i, j]] = [- ( - ( uBC_L(j) + Qi[iu[i ,j]] ) * ( uBC_L(j) +
                                                           Qi[iu[i ,j]] ) / 4 + 
                                                           ( Qi[iu[i ,j]] + Qi[iu[i+1,j ]] ) * ( Qi[iu[i ,j]] + Qi[iu[i+1,j ]] ) / 4 ) / dx - 
                                                           ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                                                        ( uBC_T(i) ) * ( vBC_T(i-1) + vBC_T(i) ) / 2 ) / dy ]
    
    # top right
    i = nx 
    j = ny 
    Qo[iu[i, j]] = [- ( - ( Qi[iu[i-1,j ]] + Qi[iu[i ,j]] ) * ( Qi[iu[i-1,j]] + Qi[iu[i ,j]] ) / 4 + 
                       ( Qi[iu[i ,j]] + uBC_R(j) ) * ( Qi[iu[i ,j]] + uBC_R(j) ) / 4 ) / dx - 
                       ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                        ( uBC_T(i) ) * ( vBC_T(i-1) + vBC_T(i) ) / 2 ) / dy] 


    # 2. V-Component
    # inner domain
    for i in range(2,1,(nx-1)):
        for j in range(3,1,(ny-1)):
            Qo[iv[i, j]] = [- ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4  + 
                               ( Qi[iu[i+1,j-1]] + Qi[iu[i+1,j ]] ) * ( Qi[iv[i ,j]] + Qi[iv[i+1,j ]] ) / 4 ) / dx - 
                               ( - ( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) * ( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) / 4 + 
                                ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) * ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) / 4 ) / dy ]
        


    


    # Edges
    # bottom inner
    j = 2 
    for i in range(2,1,(nx-1)):
        Qo[iv[i, j]] = [- ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                           ( Qi[iu[i+1,j-1]] + Qi[iu[i+1,j ]] ) * ( Qi[iv[i ,j]] + Qi[iv[i+1,j ]] ) / 4 ) / dx - 
                           ( - ( vBC_B(i) + Qi[iv[i ,j]] ) * ( vBC_B(i) + Qi[iv[i ,j]] ) / 4 + 
                            ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) * ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) / 4 ) / dy ]
    


    # top inner
    j = ny 
    for i in range(2,1,(nx-1)):
        Qo[iv[i, j]] = [- ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                           ( Qi[iu[i+1,j-1]] + Qi[iu[i+1,j ]] ) * ( Qi[iv[i ,j]] + Qi[iv[i+1,j ]] ) / 4 ) / dx  - 
                           ( - ( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) * ( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) / 4  + 
                            ( Qi[iv[i ,j]] + vBC_T(i) ) * ( Qi[iv[i ,j]] + vBC_T(i) ) / 4 ) / dy ]
    


    # left inner
    i = 1 
    for j in range(3,1,ny-1):
        Qo[iv[i, j]] = [- ( - ( uBC_L(j-1) + uBC_L(j) ) * ( vBC_L(j)) / 2 + 
                           ( Qi[iu[i+1,j-1]] + Qi[iu[i+1,j ]] ) * ( Qi[iv[i ,j]] + Qi[iv[i+1,j ]] ) / 4 ) / dx - 
                           ( - ( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) * ( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) / 4 + 
                            ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) * ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) / 4 ) / dy ]
    


    # right inner
    i = nx 
    for j in range(3,1,ny-1):
        Qo[iv[i, j]] = [- ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                           ( uBC_R(j-1) + uBC_R(j) ) * ( vBC_R(j)) / 2 ) / dx - 
                           ( - ( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) * ( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) / 4 + 
                            ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) * ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) / 4 ) / dy] 
    


    # Corners
    # bottom left
    i = 1 
    j = 2 
    Qo[iv[i, j]] = [- ( - ( uBC_L(j-1) + uBC_L(j) ) * ( vBC_L(j)) / 2 + 
                       ( Qi[iu[i+1,j-1]] + Qi[iu[i+1,j ]] ) * ( Qi[iv[i ,j]] + Qi[iv[i+1,j ]] ) / 4 ) / dx-  
                       ( - ( vBC_B(i) + Qi[iv[i ,j]] ) * ( vBC_B(i)+ Qi[iv[i ,j]] ) / 4+ 
                        ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) * ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) / 4 ) / dy ]
    # bottom right
    i = nx 
    j = 2 
    Qo[iv[i, j]] = [- ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                       ( uBC_R(j-1) + uBC_R(j) ) * ( vBC_R(j)) / 2 ) / dx- 
                       ( - ( vBC_B(i) + Qi[iv[i ,j]] ) * ( vBC_B(i)+ Qi[iv[i ,j]] ) / 4 + 
                        ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) * ( Qi[iv[i ,j]] + Qi[iv[i ,j+1]] ) / 4 ) / dy ]
    # top left
    i = 1 
    j = ny 
    Qo[iv[i, j]] = [- ( - ( uBC_L(j-1) + uBC_L(j) ) * ( vBC_L(j)) / 2 + 
                       ( Qi[iu[i+1,j-1]] + Qi[iu[i+1,j ]] ) * ( Qi[iv[i ,j]] + Qi[iv[i+1,j ]] ) / 4 ) / dx - 
                       ( - ( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) *( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) / 4 + 
                        ( Qi[iv[i ,j]] + vBC_T(i) ) * ( Qi[iv[i ,j]] + vBC_T(i) ) / 4 ) / dy ]

    # top right
    i = nx 
    j = ny 
    Qo[iv[i, j]] = [- ( - ( Qi[iu[i ,j-1]] + Qi[iu[i ,j]] ) * ( Qi[iv[i-1,j]] + Qi[iv[i ,j]] ) / 4 + 
                       ( uBC_R(j-1) + uBC_R(j) ) * ( vBC_R(j)) / 2 ) / dx - 
                       ( - ( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) *( Qi[iv[i ,j-1]] + Qi[iv[i ,j]] ) / 4 + 
                        ( Qi[iv[i ,j]] + vBC_T(i) ) * ( Qi[iv[i ,j] + vBC_T(i) ) / 4 ) / dy ]

    return Qo